# The Measurement Ceiling

The ceiling apparatus: how much of the held-out answer key is signal rather than noise, and
what fraction of that ceiling each method reaches. Seeded with the E.15 by-stand platoon
ceiling; Phase M builds out the per-stratum and level-side ceilings, the fraction-of-ceiling
table, and the 2025 refit.

Every number is read from an artifact committed under `results/phase_e`.
Nothing here trains, scores, or recomputes.

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

from src.analysis import claim1_eval
from src.model import query

## Configuration

Artifact locations and the orderings used for every table below. The scored arm is the
`d10_baseline` ensemble and the frame is 2024, which is the selection season rather than the
test season.

In [2]:
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 140)

PHASE_E_DIR = REPO_ROOT / "results/phase_e"
PHASE_C_DIR = REPO_ROOT / "results/phase_c"
PHASE_D_DIR = REPO_ROOT / "results/phase_d"
PHASE_F_DIR = REPO_ROOT / "results/phase_f"

SCORED_ARM = "e11_d10_baseline"
EVAL_SEASON = 2024

LADDER_ORDER = ["no_info_league_average", "c1_raw", "c1_bucketed", "c2_bivariate",
                "c2_book_rho_reference", "c3_gbm_outcome", "c3_gbm_full", "phase_d"]
STRATUM_ORDER = ["low", "medium", "high", "all"]


def load_e(name):
    """Read one committed Phase E artifact by filename."""
    return pd.read_csv(PHASE_E_DIR / name)


def load_json_e(name):
    """Read one committed Phase E JSON artifact by filename."""
    return json.loads((PHASE_E_DIR / name).read_text())


def load_f(name):
    """Read one committed Phase F artifact by filename."""
    return pd.read_csv(PHASE_F_DIR / name)


def load_json_f(name):
    """Read one committed Phase F summary by filename."""
    return json.loads((PHASE_F_DIR / name).read_text())


## The measurement ceiling on the platoon differential

Reliability is true talent variance over observed variance, and its square root is the
highest rank correlation any predictor could reach against this answer key. The ceiling is
low enough that the platoon result is partly a statement about the target.

In [30]:
ceiling = load_e("e15_ceiling_by_stand.csv")
ceiling.set_index("stand")[
    ["n_hitters", "tau_split_true", "mean_sampling_var_weighted",
     "reliability_variance_ratio", "reliability_mean_per_hitter",
     "ceiling_rank_corr"]].round(5)

,n_hitters,tau_split_true,mean_sampling_var_weighted,reliability_variance_ratio,reliability_mean_per_hitter,ceiling_rank_corr
stand,,,,,,
L,221,0.02706,0.00392,0.15732,0.14873,0.39663
R,324,0.02218,0.00419,0.10511,0.10245,0.32421
pooled,545,0.02430,0.00408,0.12644,0.12122,0.35559


## Between stand share of the predicted differential

The share of platoon differential variance that is the batter stand effect alone. The
observed share looks far below the model's until the sampling noise is removed from the
within stand term, and after correction the two agree. The uncorrected and corrected numbers
are the same errors in variables trap at two scales.

In [31]:
ceiling_json = load_json_e("e15_ceiling.json")
errors_in_variables = ceiling_json["part2_errors_in_variables"]
pd.Series({
    "observed variance raw": errors_in_variables["observed_variance_raw"],
    "between stand variance": errors_in_variables["between_stand_variance"],
    "within stand variance raw": errors_in_variables["within_stand_variance_raw"],
    "mean sampling variance weighted": errors_in_variables["mean_sampling_variance_weighted"],
    "within stand variance noise corrected": errors_in_variables["within_stand_variance_noise_corrected"],
    "share between stand uncorrected": errors_in_variables["share_between_stand_uncorrected"],
    "share between stand corrected": errors_in_variables["share_between_stand_corrected"],
    "corrected share CI low": errors_in_variables["ci_low"],
    "corrected share CI high": errors_in_variables["ci_high"],
    "noise share of observed variance": errors_in_variables["noise_share_of_observed_variance"],
}, name="value").to_frame().round(5)

,value
observed variance raw,0.00471
between stand variance,0.00051
within stand variance raw,0.00420
mean sampling variance weighted,0.00408
within stand variance noise corrected,0.00012
share between stand uncorrected,0.10898
share between stand corrected,0.81414
corrected share CI low,0.42175
corrected share CI high,3.49669
noise share of observed variance,0.86614


## What this notebook establishes

- The platoon measurement ceiling is low, so part of the platoon result is a statement about
  the answer key rather than about the model.